In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import re
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
from nrb_canonicalize import (
    find_candidate_duplicate_columns, coalesce_columns,
    canonicalize_line_item_regex, resolve_ambiguous_fragments,
    COMMERCIAL_BANKS, normalize_bfi_name, _PANEL_ENTITY_MAP,
)
import warnings 
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

# warnings.filterwarnings("ignore", category=DtypeWarning, module="pandas")

# 1. Robust Path Detection for both .py and .ipynb
try:
    # Works in standard .py scripts
    base_dir = Path(__file__).resolve().parent
except NameError:
    # Fallback for Jupyter Notebooks
    base_dir = Path.cwd()

# Locate data directory relative to project structure
if base_dir.name == "models":
    data_dir = base_dir.parent / "data" / "processed" / "monthly_statistical_bfis"
else:
    data_dir = base_dir / "data" / "processed" / "monthly_statistical_bfis"

csv_files = list(data_dir.glob("*.csv"))
print(f"Found {[str(i).split('/')[-1] for i in csv_files]} CSV files in: {data_dir}")
# 2. Iterate and Inspect CSVs
for file in csv_files:
    df = pd.read_csv(file)
    print(f"\nInspecting file: {file.name} {df.shape}")
    print("--" * 60)


Found ['nrb_balance_sheet.csv', 'nrb_statement_of_credit.csv', 'nrb_profit_and_loss.csv', 'nrb_bfi_list.csv'] CSV files in: /home/krishna/Documents/personal/neupane_article/network_papers/00_llama_arch/muse_llama_arch/apps/open-nrb/src/open_nrb/data/processed/monthly_statistical_bfis


/tmp/ipykernel_137329/967821476.py:35: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)



Inspecting file: nrb_balance_sheet.csv (864942, 9)
------------------------------------------------------------------------------------------------------------------------


/tmp/ipykernel_137329/967821476.py:35: DtypeWarning: Columns (2,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)



Inspecting file: nrb_statement_of_credit.csv (451124, 9)
------------------------------------------------------------------------------------------------------------------------

Inspecting file: nrb_profit_and_loss.csv (480268, 9)
------------------------------------------------------------------------------------------------------------------------

Inspecting file: nrb_bfi_list.csv (1007, 5)
------------------------------------------------------------------------------------------------------------------------


/tmp/ipykernel_137329/967821476.py:35: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


In [2]:
nrb_profit_and_loss = pd.read_csv(data_dir / "nrb_profit_and_loss.csv")
set(nrb_profit_and_loss.line_item.values.tolist())

/tmp/ipykernel_137329/3910919140.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  nrb_profit_and_loss = pd.read_csv(data_dir / "nrb_profit_and_loss.csv")


{'1 CAPITAL',
 '1 CAPITAL FUND',
 '1 LIQUID',
 '1 LIQUID FUNDS',
 '1. In Nepal Rastra',
 '1. In Nepal Rastra Bank',
 '1. Interest',
 '1. Interest Expense',
 '1. Interest Income',
 '1. Sundry',
 '1. Sundry Creditors',
 '1.1 On Deposit',
 '1.1 On Deposit Liabilities',
 '1.1 On Loans and',
 '1.1 On Loans and Advances',
 '1.1.1 Saving',
 '1.1.1 Saving Account',
 '1.1.2 Fixed',
 '1.1.2 Fixed Account',
 '1.1.2.1 Up to 3 months Fixed',
 '1.1.2.1 Up to 3 months Fixed Account',
 '1.1.2.2 3 to 6 months Fixed',
 '1.1.2.2 3 to 6 months Fixed Account',
 '1.1.2.3 6 months to 1 year Fixed',
 '1.1.2.3 6 months to 1 year Fixed Account',
 '1.1.2.4 Above 1 year Fixed',
 '1.1.2.4 Above 1 year Fixed Account',
 '1.1.3 Call',
 '1.1.3 Call Deposit',
 '1.1.4 Certificate of Deposit',
 '1.2 On',
 '1.2 On Borrowing',
 '1.2 On Investment',
 '1.2.1 Government',
 '1.2.1 Government Bonds',
 '1.2.2 Foreign',
 '1.2.2 Foreign Bonds',
 '1.2.3 NRB',
 '1.2.3 NRB Bonds',
 '1.2.4 Debenture &',
 '1.2.4 Debenture & Bonds',
 '1

In [3]:
nrb_balance_sheet = pd.read_csv(data_dir / "nrb_balance_sheet.csv")

# 1. Filter raw balance sheet data for commercial banks only
mask = nrb_balance_sheet['bfi_name'].isin(COMMERCIAL_BANKS.keys())
nrb_balance_sheet = nrb_balance_sheet[mask].copy()

nrb_balance_sheet['bfi_name'] = nrb_balance_sheet['bfi_name'].map(normalize_bfi_name)
nrb_balance_sheet['line_item'] = nrb_balance_sheet['line_item'].apply(canonicalize_line_item_regex)
nrb_balance_sheet = resolve_ambiguous_fragments(nrb_balance_sheet)
nrb_balance_sheet['value_mn_npr'] = pd.to_numeric(nrb_balance_sheet['value_mn_npr'], errors='coerce')

# 2. drop junk rows before pivoting, so they can't become spurious columns
drop_labels = {'DROP_TOTAL_ROW', 'DROP_FRAGMENT', 'UNKNOWN'}
nrb_balance_sheet = nrb_balance_sheet[
    ~nrb_balance_sheet['line_item'].isin(drop_labels)
    & ~nrb_balance_sheet['line_item'].str.startswith('UNKNOWN SECTION >', na=False)
]

pivot = nrb_balance_sheet.pivot_table(index=['year_bs', 'month_bs', 'bfi_name'],
                                        columns='line_item', values='value_mn_npr',
                                        aggfunc='sum').reset_index()

from nrb_canonicalize import build_paper1_features
paper1_features = build_paper1_features(pivot)

legacy_cols = [
    '2.f Borrowings - Govt Non-Financial Institutions',
    '2.g Borrowings - Other Non-Financial Institutions',
    '1.c Proposed Bonus Share',
    '1.a.1 Cash Balance - Cash in Transit',
    '6 LOANS AGAINST COLLECTED BILLS',
    '6 RECONCILIATION ACCOUNT',
    '6.a Loans Against Collected Bills - Domestic/Foreign',
    '8 CALLS IN ADVANCE',
    '2.d Other Financial Institutions',
    '2.e Bonds and Securities / NRB Bonds',
    '9 EXPENSES NOT WRITTEN OFF',
    '4 LOANS & ADVANCES > FINANCIAL',
    '4 LOANS & ADVANCES > GOVERNMENT',
    '3.e Deposits - Non Residents',
]
pivot = pivot.drop(columns=[c for c in legacy_cols if c in pivot.columns])

paper1_features


/tmp/ipykernel_137329/3456608721.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  nrb_balance_sheet = pd.read_csv(data_dir / "nrb_balance_sheet.csv")


RAW COLUMNS: ['7 FIXED ASSETS', 'INTERBANK LENDING (Marginal r_i)', '3 SHARE & OTHER INVESTMENT', '2 INVESTMENT IN SECURITIES', '4 LOANS & ADVANCES', '6 LOANS AGAINST COLLECTED BILLS', '10 NON BANKING ASSETS', '5 BILL PURCHASED', '9 EXPENSES NOT WRITTEN OFF', '8 OTHER ASSETS', '1 LIQUID FUNDS'],'1 CAPITAL FUND',           '1 LIQUID FUNDS', '4 LOANS & ADVANCES', '3 DEPOSITS', ['2.g Borrowings - Other Non-Financial Institutions', '2.f Borrowings - Govt Non-Financial Institutions', '2.e Bonds and Securities / NRB Bonds', '2.b Interbank Borrowing', '2.c Foreign Banks and Financial Institutions', '2.d Other Financial Institutions', '2.a Borrowings - NRB'],              'INTERBANK LENDING (Marginal r_i)','2 BORROWINGS',             '2.a Borrowings - NRB', '1 CAPITAL FUND'


line_item,year_bs,month_bs,bfi_name,size,capital_ratio,liquidity_ratio,loan_to_deposit,nrb_dependence,r_i,c_i,K_i
0,2070,Asar,ADBNL,91042.88,0.144274,0.106055,1.010334,0.997756,NaN,NaN,13135.15
1,2070,Asar,BOK_LUMBINI,49705.82,0.104796,0.170702,0.806311,0.826604,NaN,NaN,5208.95
2,2070,Asar,CTBNL,12275.73,0.166061,0.175044,0.891997,0.857772,NaN,NaN,2038.52
3,2070,Asar,Century,11104.83,0.104071,0.108714,0.799610,1.000000,NaN,NaN,1155.69
4,2070,Asar,Citizens,26910.66,0.084790,0.179298,0.778178,0.816798,NaN,NaN,2281.76
5,2070,Asar,Civil,19246.00,0.115184,0.189197,0.785094,0.892037,NaN,NaN,2216.84
6,2070,Asar,EBL,66866.41,0.062299,0.167734,0.762440,0.888036,NaN,NaN,4165.74
7,2070,Asar,Global,41716.49,0.081649,0.127524,0.786109,0.796086,NaN,NaN,3406.12
8,2070,Asar,HBL,62911.90,0.073627,0.090762,0.747069,0.902936,NaN,NaN,4632.01
9,2070,Asar,Janata,15823.54,0.130221,0.130507,0.838671,0.939190,NaN,NaN,2060.55
